# Lab Assignment: Data Quality & Data Cleaning for Data Warehouse

**สถานการณ์:** คุณเป็น Data Engineer ของร้านค้าออนไลน์ และได้รับไฟล์ `e_commerce_raw.csv` ที่ต้องเตรียมก่อนโหลดเข้าสู่ Data Warehouse  
**เป้าหมาย:** สำรวจปัญหาคุณภาพข้อมูล ออกแบบกฎการทำความสะอาด และส่งออกข้อมูลที่พร้อมใช้งาน

> ห้ามลบข้อมูลผิดปกติโดยไม่อธิบายเหตุผล ต้องพิจารณา Business Context โดยเฉพาะรายการโปรโมชัน


## ผลลัพธ์การเรียนรู้
1. ตรวจสอบข้อมูลด้วย `info()`, `isnull()`, `describe()` และ `duplicated()`
2. เชื่อมโยงปัญหากับ 6 มิติ: Accuracy, Completeness, Consistency, Uniqueness, Timeliness, Validity
3. จัดการ Missing Values และรูปแบบข้อมูลที่ไม่สอดคล้อง
4. ตรวจจับ Outlier ด้วย IQR และทำ Capping อย่างมีเหตุผล
5. สร้างไฟล์ Clean Data และรายงาน Data Quality ก่อน–หลัง


## 0) เตรียมระบบและอ่านข้อมูล

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'e_commerce_raw.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'student_submission'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print('Shape:', df.shape)
df.head()

## 1) Data Profiling

ให้แสดงผลต่อไปนี้และเขียนข้อสังเกตใต้เซลล์
- โครงสร้างและชนิดข้อมูล
- จำนวน Missing Values รายคอลัมน์
- สถิติเชิงพรรณนาของข้อมูลตัวเลข
- จำนวนแถวซ้ำทั้งแถว
- จำนวนค่าที่แตกต่างในคอลัมน์สำคัญ


In [ ]:
# TODO 1: Data Profiling
print('--- Shape of dataset ---')
print(df.shape)

print('\n--- Columns and data types ---')
df.info()

print('\n--- First 5 rows ---')
display(df.head())

print('\n--- Missing values per column ---')
print(df.isnull().sum())

print('\n--- Number of exact duplicates ---')
print(df.duplicated().sum())

print('\n--- Descriptive statistics ---')
display(df.describe(include='all').T)


### คำถาม 1
ระบุปัญหาที่พบอย่างน้อย **5 ปัญหา** และจับคู่กับมิติ Data Quality อย่างน้อย **3 มิติ**

| ปัญหาที่พบ | คอลัมน์ | มิติ Data Quality | หลักฐาน/จำนวนแถว |
|---|---|---|---|
| ข้อมูลอีเมลของลูกค้าสูญหาย (Missing values) | Customer_Email | Completeness | พบค่าว่าง (Null) ทั้งหมด 50 แถว |
| ข้อมูลจังหวัดของลูกค้าสูญหาย (Missing values) | Province | Completeness | พบค่าว่าง (Null) ทั้งหมด 12 แถว |
| รูปแบบข้อมูลเพศมีความไม่สอดคล้องกัน | Gender | Consistency | มีการป้อนข้อมูลหลายแบบ (female, FEMALE, f, Female, male, MALE, m, Male, male ) |
| รูปแบบวิธีชำระเงินมีความไม่สอดคล้องกัน | Payment_Method | Consistency | มีการป้อนข้อมูลหลากหลาย (Transfer, COD, Credit Card, promptpay, PP, CC, Card ฯลฯ) |
| รูปแบบวันที่สั่งซื้อมีหลากหลายและบางค่าระบุวันที่ผิดพลาด | Order_Date | Consistency / Validity | มีรูปแบบ YYYY-MM-DD, DD/MM/YYYY, MM/DD/YYYY, YYYY/MM/DD และมีค่าผิดเช่น 31/02/2026, 2026-13-05 และข้อความที่ไม่ใช่วันที่ |
| ธุรกรรมซ้ำซ้อนในคลังข้อมูล (Exact Duplicates) | ทุกคอลัมน์ | Uniqueness | พบแถวซ้ำกันทุกคอลัมน์ (Exact Duplicate rows) จำนวน 18 แถว |
| จำนวนสินค้าที่สั่งซื้อผิดปกติและติดลบ | Quantity | Validity | มีจำนวนสินค้าเป็น 0, -3 และ 99 ซึ่งอยู่นอกเกณฑ์ปกติ 1-20 จำนวน 3 แถว |
| ยอดขายสุทธิไม่สอดคล้องกับปริมาณ ราคา และส่วนลด | Transaction_Amount | Accuracy | ยอดไม่ตรงตามสูตรคำนวณจริงจำนวน 51 แถว |


## 2) Completeness: จัดการ Missing Values

In [ ]:
# TODO 2.1: ตรวจสอบ Customer_Email และ Province ที่หายไป
print('Missing Email count:', df['Customer_Email'].isnull().sum())
print('Missing Province count:', df['Province'].isnull().sum())

# TODO 2.2: ออกแบบวิธีเติม Customer_Email โดยต้องไม่ทำให้แถว Fact หาย
df['Customer_Email_Was_Missing'] = df['Customer_Email'].isnull()
df['Customer_Email'] = df.apply(
    lambda r: f"unknown_{r['Customer_ID']}@unknown.local" if pd.isnull(r['Customer_Email']) else r['Customer_Email'],
    axis=1
)

# TODO 2.3: เติม Province ที่หายเป็น 'Unknown' พร้อม flag
df['Province_Was_Missing'] = df['Province'].isnull()
df['Province'] = df['Province'].fillna('Unknown')

print('\nAfter cleaning missing values:')
print('Missing Email count:', df['Customer_Email'].isnull().sum())
print('Missing Province count:', df['Province'].isnull().sum())


### คำถาม 2
อธิบายว่าเหตุใดการใช้ `dropna()` กับ `Customer_Email` อาจทำให้ยอดขายรวมใน Data Warehouse ผิดเพี้ยน และเหตุใดการเติมค่าเฉลี่ยจึงไม่เหมาะกับข้อมูลประเภทอีเมล

**คำตอบ:**
1. **ผลของการใช้ `dropna()`:** การใช้ `dropna()` ลบแถวที่ไม่มีอีเมลทิ้ง จะส่งผลให้แถวธุรกรรม (Fact rows) ของยอดซื้อขายที่เกี่ยวข้องกับลูกค้ารายนั้นหายไปทั้งหมด ส่งผลให้ยอดขายรวมและจำนวนธุรกรรมรวมของระบบลดลงต่ำกว่าความเป็นจริง ซึ่งขัดต่อวัตถุประสงค์ในการวิเคราะห์ยอดขายรวมในคลังข้อมูลที่ต้องการความครบถ้วนของประวัติธุรกรรม (Financial Fact)
2. **เหตุผลที่การเติมค่าเฉลี่ยไม่เหมาะสมกับอีเมล:** อีเมลเป็นข้อมูลระบุตัวตน (Identifier) หรือข้อมูลประเภทจัดกลุ่ม (Categorical/Text) ซึ่งไม่มีคุณสมบัติเชิงตัวเลขเชิงปริมาณ การนำข้อมูลที่เป็นตัวอักษรมาหาค่าเฉลี่ยทางคณิตศาสตร์ไม่สามารถทำได้และไม่มีความหมายในทางปฏิบัติ


## 3) Consistency: จัดรูปแบบข้อมูลให้เป็นมาตรฐาน

In [ ]:
# TODO 3.1: Standardize Gender ให้เหลือ M, F, Unknown
gender_map = {
    'female': 'F', 'FEMALE': 'F', 'f': 'F', 'Female': 'F',
    'male': 'M', 'MALE': 'M', 'm': 'M', 'Male': 'M', 'male ': 'M'
}
df['Gender'] = df['Gender'].str.strip().map(lambda x: gender_map.get(x, 'Unknown'))

# TODO 3.2: Standardize Payment_Method ให้เหลือ
# Credit Card, PromptPay, Cash on Delivery, Bank Transfer, Unknown
def clean_payment_method(pm):
    pm_clean = str(pm).strip().lower()
    if pm_clean in ['credit card', 'cc', 'card']:
        return 'Credit Card'
    elif pm_clean in ['promptpay', 'prompt pay', 'pp']:
        return 'PromptPay'
    elif pm_clean in ['cash on delivery', 'cod', 'cash']:
        return 'Cash on Delivery'
    elif pm_clean in ['bank transfer', 'transfer', 'bank_transfer']:
        return 'Bank Transfer'
    else:
        return 'Unknown'

df['Payment_Method'] = df['Payment_Method'].apply(clean_payment_method)

# TODO 3.3: แปลง Order_Date ซึ่งมีหลายรูปแบบให้เป็น YYYY-MM-DD
# ระวังความกำกวมระหว่าง DD/MM/YYYY และ MM/DD/YYYY
# ใช้ Load_Timestamp เป็นข้อมูลประกอบ และสร้าง flag สำหรับค่าที่ต้อง fallback/impute
def parse_order_date(row):
    od_str = str(row['Order_Date']).strip()
    lt_str = str(row['Load_Timestamp']).strip()
    lt = pd.to_datetime(lt_str)
    lt_date_str = lt.strftime('%Y-%m-%d')
    
    # 1. ลองแปลงโดยตรงแบบ YYYY-MM-DD
    for fmt in ['%Y-%m-%d', '%Y/%m/%d']:
        try:
            d = pd.to_datetime(od_str, format=fmt, errors='raise')
            return d.strftime('%Y-%m-%d'), 'YYYY-MM-DD', False
        except:
            pass
            
    # 2. ลองแปลงแบบ DD/MM/YYYY และ MM/DD/YYYY
    fmts_dmy = ['%d-%m-%Y', '%d/%m/%Y']
    fmts_mdy = ['%m-%d-%Y', '%m/%d/%Y']
    
    d_dmy = None
    for fmt in fmts_dmy:
        try:
            d_dmy = pd.to_datetime(od_str, format=fmt, errors='raise')
            break
        except:
            pass
            
    d_mdy = None
    for fmt in fmts_mdy:
        try:
            d_mdy = pd.to_datetime(od_str, format=fmt, errors='raise')
            break
        except:
            pass
            
    # เปรียบเทียบกับ Load_Timestamp
    if d_dmy is not None and d_dmy.strftime('%Y-%m-%d') == lt_date_str:
        return d_dmy.strftime('%Y-%m-%d'), 'DD/MM/YYYY', False
    if d_mdy is not None and d_mdy.strftime('%Y-%m-%d') == lt_date_str:
        return d_mdy.strftime('%Y-%m-%d'), 'MM/DD/YYYY', False
        
    # หากมีรูปแบบใดรูปแบบหนึ่งใช้ได้เพียงแบบเดียว
    if d_dmy is not None and d_mdy is None:
        return d_dmy.strftime('%Y-%m-%d'), 'DD/MM/YYYY', False
    if d_mdy is not None and d_dmy is None:
        return d_mdy.strftime('%Y-%m-%d'), 'MM/DD/YYYY', False
        
    # หากแปลงได้ทั้งสองแบบ แต่ไม่ตรงกับ load timestamp พอดี ให้ตรวจสอบความสมเหตุสมผลเชิงเวลา (สั่งซื้อต้องเกิดก่อนหรือพร้อมวันโหลดข้อมูล)
    if d_dmy is not None and d_mdy is not None:
        lt_date = lt.date()
        diff_dmy = (lt_date - d_dmy.date()).days
        diff_mdy = (lt_date - d_mdy.date()).days
        
        if diff_dmy >= 0 and diff_mdy < 0:
            return d_dmy.strftime('%Y-%m-%d'), 'DD/MM/YYYY', False
        if diff_mdy >= 0 and diff_dmy < 0:
            return d_mdy.strftime('%Y-%m-%d'), 'MM/DD/YYYY', False
            
        # หากเป็นไปได้ทั้งคู่ เลือกวันที่ใกล้กับวันโหลดข้อมูลมากกว่า
        if diff_dmy >= 0 and diff_mdy >= 0:
            if diff_dmy <= diff_mdy:
                return d_dmy.strftime('%Y-%m-%d'), 'DD/MM/YYYY', False
            else:
                return d_mdy.strftime('%Y-%m-%d'), 'MM/DD/YYYY', False
                
    # 3. Fallback หากแปลงไม่ได้เลย หรือข้อมูลมีข้อผิดพลาด
    return lt_date_str, 'Fallback to Load_Timestamp', True

res = df.apply(parse_order_date, axis=1)
df['Order_Date_Cleaned'] = [r[0] for r in res]
df['Order_Date_Parse_Method'] = [r[1] for r in res]
df['Order_Date_Was_Imputed'] = [r[2] for r in res]
df['Order_Date'] = df['Order_Date_Cleaned']
df = df.drop(columns=['Order_Date_Cleaned'])

print('Gender Counts:')
print(df['Gender'].value_counts())
print('\nPayment Method Counts:')
print(df['Payment_Method'].value_counts())
print('\nOrder Date Imputed Rows:', df['Order_Date_Was_Imputed'].sum())


### คำถาม 3
ยกตัวอย่าง 1 วันที่สามารถตีความได้ทั้ง DD/MM/YYYY และ MM/DD/YYYY แล้วอธิบายว่าคุณใช้หลักฐานใดช่วยตัดสิน

**คำตอบ:**
ตัวอย่างคือ ค่าวันที่ `'02/01/2026'`  
- หากตีความเป็น `DD/MM/YYYY` จะเป็นวันที่ **2 มกราคม 2026**
- หากตีความเป็น `MM/DD/YYYY` จะเป็นวันที่ **1 กุมภาพันธ์ 2026**  

**หลักฐานที่ใช้ตัดสิน:**  
เราเปรียบเทียบกับฟิลด์ `Load_Timestamp` ของแถวนั้น ซึ่งระบุวันเวลาที่ข้อมูลเข้าระบบเป็น `'2026-02-01 18:00:00'` (1 กุมภาพันธ์ 2026) ทำให้สรุปได้ว่ารูปแบบการเขียนคือ `MM/DD/YYYY` (1 กุมภาพันธ์ 2026) เพราะในโลกธุรกิจจริง ธุรกรรมการซื้อขายมักจะถูกสร้างและส่งเข้าสู่ระบบในเวลาใกล้เคียงกัน


## 4) Uniqueness: ข้อมูลซ้ำและความซ้ำซ้อนเชิงความหมาย

In [ ]:
# TODO 4.1: นับและลบ Exact Duplicates โดยเก็บแถวแรก
print('Row count before dropping duplicates:', len(df))
exact_duplicates = df.duplicated().sum()
print('Exact duplicates count:', exact_duplicates)
df = df.drop_duplicates(keep='first').copy()
print('Row count after dropping duplicates:', len(df))

# TODO 4.2 (Bonus): ตรวจหาลูกค้าที่ Customer_Email เดียวกัน แต่ Customer_Name เขียนต่างกัน
# ค้นหา Near Duplicate
dup_names = df.groupby('Customer_Email')['Customer_Name'].nunique()
mismatched_emails = dup_names[dup_names > 1].index
print('\nNear Duplicates (Same Email, Different Name Spellings):')
for email in mismatched_emails:
    names = df[df['Customer_Email'] == email]['Customer_Name'].unique()
    c_id = df[df['Customer_Email'] == email]['Customer_ID'].unique()
    print(f"Customer ID: {c_id}, Email: {email} -> Names found: {names}")


### คำถาม 4 (อธิบายข้อจำกัดของ Exact Match และ Near Duplicate)
อธิบายว่าเหตุใด `drop_duplicates()` เพียงอย่างเดียวจึงตรวจไม่พบ Near Duplicate และแนวทาง Fuzzy Matching ในระบบจริง

**คำตอบ:**
1. **เหตุผลที่ `drop_duplicates()` ตรวจไม่พบ:** ฟังก์ชัน `drop_duplicates()` ทำงานบนหลักการเปรียบเทียบค่าแบบเท่ากันทุกอักขระ (Exact Matching) บนสตริง หากมีสะกดผิดเล็กน้อย เช่น เว้นวรรคเกินอย่างตัวอย่าง `'Ploy Saelim'` กับ `'Ploy  Saelim'` (เว้นวรรค 2 ตัว) ถือว่าเป็นอักขระคนละชุดกัน ทำให้ฟังก์ชันไม่ถือว่าเป็นแถวที่ซ้ำกัน
2. **แนวทาง Fuzzy Matching ในระบบจริง:** ในการทำงานจริงกับข้อมูลขนาดใหญ่ เราจะใช้เทคนิคการคำนวณระยะห่างระหว่างข้อความ เช่น **Levenshtein Distance** หรือ **Jaro-Winkler Similarity** เพื่อหาความคล้ายคลึงของข้อความ และตั้งค่าเกณฑ์ความเชื่อมั่น (Similarity Threshold เช่น >= 90%) ร่วมกับการใช้ Surrogate Key หรือข้อมูลอื่นมาเชื่อมโยงกัน


In [ ]:
# TODO 5.1: แปลง Quantity, Unit_Price, Discount_Rate และ Transaction_Amount เป็นตัวเลข
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')
df['Discount_Rate'] = pd.to_numeric(df['Discount_Rate'], errors='coerce')
df['Transaction_Amount'] = pd.to_numeric(df['Transaction_Amount'], errors='coerce')

# TODO 5.2: ตรวจ Quantity ต้องอยู่ในช่วง 1-20
df['Quantity_Was_Invalid'] = ~df['Quantity'].between(1, 20)
print('Invalid quantity count:', df['Quantity_Was_Invalid'].sum())

# TODO 5.3: คำนวณ Expected_Amount = Quantity * Unit_Price * (1 - Discount_Rate)
# สร้าง Accuracy_Flag เมื่อผลต่างเกิน 1 บาท
df['Expected_Amount'] = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount_Rate'])
df['Accuracy_Flag'] = (df['Transaction_Amount'] - df['Expected_Amount']).abs() > 1.0
print('Transaction Amount mismatches before correction:', df['Accuracy_Flag'].sum())

# TODO 5.4: Standardize Order_Status และตรวจ Validity
def standardize_status(status):
    status_clean = str(status).strip().lower()
    if status_clean in ['completed', 'done']:
        return 'Completed'
    elif status_clean in ['cancelled', 'cancel']:
        return 'Cancelled'
    elif status_clean in ['processing']:
        return 'Processing'
    elif status_clean in ['returned']:
        return 'Returned'
    else:
        return 'Unknown'

df['Order_Status'] = df['Order_Status'].apply(standardize_status)
df['Order_Status_Was_Invalid'] = ~df['Order_Status'].isin(['Completed', 'Processing', 'Cancelled', 'Returned'])
print('Invalid Order Status count:', df['Order_Status_Was_Invalid'].sum())

# แก้ไขยอดสำหรับรายการที่ไม่ใช่โปรโมชัน
mask_correct = (df['Is_Promotion'] != 'Y') & df['Accuracy_Flag']
df.loc[mask_correct, 'Transaction_Amount'] = df.loc[mask_correct, 'Expected_Amount']

# อัปเดต Accuracy_Flag หลังแก้ค่า
df['Accuracy_Flag'] = (df['Transaction_Amount'] - df['Expected_Amount']).abs() > 1.0
print('Transaction Amount mismatches after correction:', df['Accuracy_Flag'].sum())


### คำถาม 4
รายการที่ `Transaction_Amount` ไม่ตรงกับสูตรควรแก้ไขทุกแถวหรือไม่? ให้อธิบายผลของ `Is_Promotion` ต่อการตัดสินใจ

**คำตอบ:**
ไม่ควรแก้ไขทุกแถวโดยอัตโนมัติ เนื่องจากต้องคำนึงถึงบริบททางธุรกิจ (Business Context):
1. **รายการโปรโมชัน (`Is_Promotion = 'Y'`):** หากยอดการชำระเงินไม่ตรงกับสูตรมาตรฐาน อาจเกิดจากการใช้คูปองส่วนลดพิเศษเพิ่มเติมหรือโปรโมชันร่วมรายการ ซึ่งเป็นการซื้อขายจริงที่มีเงื่อนไขธุรกิจเฉพาะ การแก้ไขยอดข้อมูลในแถวดังกล่าวให้ตรงตามสูตรปกติโดยพลการจะทำให้สูญเสียข้อมูลสำคัญนี้ไป
2. **รายการปกติ (`Is_Promotion = 'N'`):** หากมีส่วนต่างเกิดขึ้น ถือว่าเป็นความผิดพลาดของระบบคำนวณหรือข้อผิดพลาดตอนกรอกข้อมูล (Data Entry/System Error) ซึ่งควรได้รับการแก้ไขให้ตรงตามความจริง


## 6) Outliers: IQR และ Capping/Clipping

In [ ]:
# TODO 6.1: ใช้เฉพาะรายการที่ Is_Promotion != 'Y' คำนวณ Q1, Q3, IQR, Lower Bound, Upper Bound
non_promo = df[df['Is_Promotion'] != 'Y']
Q1 = non_promo['Transaction_Amount'].quantile(0.25)
Q3 = non_promo['Transaction_Amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR: {IQR}")
print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")

# TODO 6.2: สร้าง Outlier Flag ก่อนทำ Capping
df['Outlier_Flag_Before_Capping'] = (df['Transaction_Amount'] < lower_bound) | (df['Transaction_Amount'] > upper_bound)
print('Outliers count before capping:', df['Outlier_Flag_Before_Capping'].sum())

# TODO 6.3: ใช้ .clip() กับ Transaction_Amount เฉพาะรายการที่ไม่ใช่โปรโมชัน
df.loc[df['Is_Promotion'] != 'Y', 'Transaction_Amount'] = df.loc[df['Is_Promotion'] != 'Y', 'Transaction_Amount'].clip(lower_bound, upper_bound)

# TODO 6.4: วาด Boxplot ก่อนและหลัง Capping
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.boxplot(df_raw['Transaction_Amount'].dropna())
plt.title('Transaction Amount (Raw - Before Capping)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.boxplot(df['Transaction_Amount'])
plt.title('Transaction Amount (Cleaned - After Capping)')
plt.grid(True)
plt.show()


### คำถาม 5
เพราะเหตุใดข้อมูลที่อยู่นอก IQR Bound จึงไม่จำเป็นต้องเป็นข้อมูลผิดเสมอ? ยกตัวอย่างจาก Dataset นี้

**คำตอบ:**
ข้อมูลที่เป็น Outlier ตามหลักสถิติไม่ได้หมายความว่าเป็นข้อมูลผิดพลาดเสมอไป แต่อาจเป็นข้อมูลจริงที่มีค่าสูงมากเป็นพิเศษ ตัวอย่างจากชุดข้อมูลนี้คือ **ยอดขายที่เป็นโปรโมชัน (`Is_Promotion = 'Y'`)** ซึ่งอาจเป็นการสั่งซื้อสินค้าขนาดใหญ่หรือลูกค้ารายใหญ่ (VIP/Wholesale) ที่ซื้อสินค้าปริมาณมากในราคาโปรโมชัน การตัดค่าเหล่านี้ทิ้งจะทำให้เราสูญเสียข้อมูลสำคัญเชิงลึกทางธุรกิจ (Business Insight)


## 7) Timeliness: ความทันเวลาในการโหลดข้อมูล

In [ ]:
# TODO 7: คำนวณ Load_Delay_Hours จาก Order_Date ถึง Load_Timestamp
df['Order_Date_dt'] = pd.to_datetime(df['Order_Date'])
df['Load_Timestamp_dt'] = pd.to_datetime(df['Load_Timestamp'])
df['Load_Delay_Hours'] = (df['Load_Timestamp_dt'] - df['Order_Date_dt']).dt.total_seconds() / 3600.0
df['Is_Late_Load'] = df['Load_Delay_Hours'] > 24.0

print('Late Load Transactions:', df['Is_Late_Load'].sum())
print('Percentage of Late Loads:', (df['Is_Late_Load'].mean() * 100).round(2), '%')

# ลบคอลัมน์ชั่วคราวออก
df = df.drop(columns=['Order_Date_dt', 'Load_Timestamp_dt'])


## 8) Data Quality Report และ Export

In [ ]:
# TODO 8.1: สรุปตัวชี้วัดก่อนและหลัง Cleaning เป็น DataFrame
raw_emails_missing = df_raw['Customer_Email'].isnull().sum()
raw_exact_duplicates = exact_duplicates
invalid_dates_before = df['Order_Date_Was_Imputed'].sum()
invalid_quantities_before = (~pd.to_numeric(df_raw['Quantity'], errors='coerce').between(1, 20)).sum()

expected_amount_raw = pd.to_numeric(df_raw['Quantity'], errors='coerce') * pd.to_numeric(df_raw['Unit_Price'], errors='coerce') * (1 - pd.to_numeric(df_raw['Discount_Rate'], errors='coerce'))
mismatched_amounts_before = ((pd.to_numeric(df_raw['Transaction_Amount'], errors='coerce') - expected_amount_raw).abs() > 1.0).sum()
invalid_statuses_before = (~df_raw['Order_Status'].str.strip().str.lower().isin(['completed', 'done', 'cancelled', 'cancel', 'processing', 'returned'])).sum()
outliers_capped_before = df['Outlier_Flag_Before_Capping'].sum()
late_loads_before = df['Is_Late_Load'].sum()

metrics = {
    'Metric': [
        'Total Rows',
        'Missing Emails',
        'Exact Duplicates',
        'Invalid Dates (Imputed)',
        'Invalid Quantities',
        'Mismatched Amounts',
        'Invalid Statuses',
        'Outliers Capped',
        'Late Loads'
    ],
    'Before': [
        len(df_raw),
        raw_emails_missing,
        raw_exact_duplicates,
        invalid_dates_before,
        invalid_quantities_before,
        mismatched_amounts_before,
        invalid_statuses_before,
        outliers_capped_before,
        late_loads_before
    ],
    'After': [
        len(df),
        0, # Missing Emails (filled with placeholders)
        0, # Exact Duplicates (dropped)
        0, # Invalid Dates (imputed)
        df['Quantity_Was_Invalid'].sum(), # Invalid Quantities (flagged)
        df['Accuracy_Flag'].sum(), # Mismatched Amounts (7 promotion ones remain)
        df['Order_Status_Was_Invalid'].sum(), # Invalid Statuses (standardized/flagged)
        (df['Transaction_Amount'].apply(lambda x: x < lower_bound or x > upper_bound)).sum(), # Outliers remaining (promotion ones)
        df['Is_Late_Load'].sum() # Late Loads
    ]
}
report_df = pd.DataFrame(metrics)
display(report_df)

# TODO 8.2: Export
df.to_csv(OUTPUT_DIR / 'e_commerce_clean.csv', index=False, encoding='utf-8-sig')
report_df.to_csv(OUTPUT_DIR / 'data_quality_report.csv', index=False, encoding='utf-8-sig')
print('Data and Report exported successfully!')


## 9) Reflection
ตอบเป็นย่อหน้าสั้น ๆ
1. **ขั้นตอนใดมีความเสี่ยงทำให้ข้อมูลจริงสูญหายมากที่สุด?**
   - ขั้นตอนการจัดการความซ้ำซ้อนและการกรองข้อมูล (Uniqueness / Duplicates) และการกำจัด Outliers มีความเสี่ยงในการทำข้อมูลจริงสูญหายมากที่สุด โดยเฉพาะการใช้ `drop_duplicates()` หรือการกรอง/ลบค่าที่อยู่จำกัดนอกขอบเขต (IQR Bounds) โดยไม่ตรวจสอบประเภทธุรกรรมหรือลักษณะธุรกิจ (เช่น การลบยอดซื้อที่มีค่าสูงพิเศษของกลุ่มลูกค้าโปรโมชัน) ซึ่งจะส่งผลเสียต่อความสมบูรณ์และถูกต้องของข้อมูลขายจริงในระบบคลังข้อมูล

2. **กฎใดควรนำไปทำ Automation ใน Data Pipeline?**
   - กฎการทำความสะอาดประเภทข้อมูลเชิงโครงสร้างและค่าที่ขาดหายควรนำไปทำ Automation เช่น: (1) การเติม Missing value ด้วย traceable placeholders สำหรับอีเมล (2) การจัดมาตรฐานข้อมูลเพศ (Standardize Gender) และวิธีจ่ายเงิน (Payment Method) ด้วยพจนานุกรมคำพ้อง (3) การคำนวณและตรวจสอบเงื่อนไขความถูกต้องของเงิน `Transaction_Amount = Quantity * Unit_Price * (1 - Discount_Rate)` และการแจ้งเตือนเมื่อเกิดส่วนต่างสำหรับแถวทั่วไปที่ไม่ใช่โปรโมชัน

3. **หากใช้ Great Expectations คุณจะสร้าง Expectation ใดอย่างน้อย 3 ข้อ?**
   - `expect_column_values_to_be_between("Quantity", min_value=1, max_value=20)` เพื่อควบคุมความถูกต้องของจำนวนสินค้าต่อการซื้อ
   - `expect_column_values_to_match_regex("Customer_Email", regex=r'^[^@]+@[^@]+\.[^@]+$')` เพื่อควบคุมรูปแบบความถูกต้องของฟิลด์อีเมล
   - `expect_column_values_to_be_in_set("Order_Status", ["Completed", "Processing", "Cancelled", "Returned"])` เพื่อยืนยันว่าสถานะธุรกรรมต้องผ่านการทำความสะอาดและมีค่าตรงตามข้อตกลงที่กำหนดไว้เสมอ


## Checklist ก่อนส่ง
- [ ] Notebook รันจากบนลงล่างได้โดยไม่ Error
- [ ] มีคำตอบคำถามทั้ง 5 ข้อและ Reflection
- [ ] ส่ง `e_commerce_clean.csv`
- [ ] ส่ง `data_quality_report.csv`
- [ ] ไม่ลบรายการโปรโมชันเพียงเพราะเป็น Outlier
